# 04 — Engineer Time Series
Aggregates scored articles into weekly time series per subtopic.
Input: `data/scored_articles.csv` → Output: `data/timeseries.csv`

In [1]:
import pandas as pd

df = pd.read_csv('data/scored_articles.csv', parse_dates=['date'])
df = df.sort_values('date')
print(df.shape)

(24584, 11)


In [2]:
# Resample per week per subtopic
df['week'] = df['date'].dt.to_period('W').dt.start_time

ts = (
    df.groupby(['week', 'subtopic'])
    .agg(
        article_count=('id', 'count'),
        sentiment_mean=('sentiment', 'mean'),
        sentiment_std=('sentiment', 'std'),
        positive_ratio=('sentiment_label', lambda x: (x == 'positive').mean()),
        negative_ratio=('sentiment_label', lambda x: (x == 'negative').mean()),
    )
    .reset_index()
)

ts['sentiment_std'] = ts['sentiment_std'].fillna(0)
print(ts.shape)
ts.head(10)

(1252, 7)


,week,subtopic,article_count,sentiment_mean,sentiment_std,positive_ratio,negative_ratio
0,2021-12-27,AI Regulation,3,-0.083567,0.794265,0.333333,0.666667
1,2021-12-27,Other,16,0.148369,0.573489,0.687500,0.312500
2,2022-01-03,AI & Big Tech,2,0.591850,0.240911,1.000000,0.000000
3,2022-01-03,AI & Jobs,1,0.216800,0.000000,1.000000,0.000000
4,2022-01-03,AI Regulation,10,-0.114640,0.645565,0.400000,0.500000
5,2022-01-03,AI in Healthcare,2,-0.351450,0.644811,0.500000,0.500000
6,2022-01-03,Other,52,0.111523,0.580889,0.538462,0.346154
7,2022-01-10,AI & Big Tech,1,-0.493900,0.000000,0.000000,1.000000
8,2022-01-10,AI Regulation,9,-0.383700,0.505733,0.333333,0.666667
9,2022-01-10,AI in Healthcare,2,-0.418400,0.271670,0.000000,1.000000


In [3]:
# Pivot to wide format for easy plotting (optional)
# Each subtopic becomes a column group
ts_wide = ts.pivot_table(index='week', columns='subtopic', values='article_count', fill_value=0)
print('Wide format shape:', ts_wide.shape)
ts_wide.head()

Wide format shape: (210, 7)


subtopic,AI & Big Tech,AI & Jobs,AI Regulation,AI Safety,AI in Healthcare,Generative AI,Other
week,,,,,,,
2021-12-27,0.0,0.0,3.0,0.0,0.0,0.0,16.0
2022-01-03,2.0,1.0,10.0,0.0,2.0,0.0,52.0
2022-01-10,1.0,0.0,9.0,0.0,2.0,0.0,60.0
2022-01-17,2.0,0.0,14.0,0.0,4.0,1.0,48.0
2022-01-24,1.0,2.0,13.0,0.0,1.0,1.0,54.0


In [4]:
ts.to_csv('data/timeseries.csv', index=False)
ts_wide.to_csv('data/timeseries_wide.csv')
print('Saved: data/timeseries.csv and data/timeseries_wide.csv')

Saved: data/timeseries.csv and data/timeseries_wide.csv
